In [ ]:
from pathlib import Path
import json
import numpy as np
from typing import List, Tuple
import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import math
from pathlib import Path
from typing import Dict
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from collections import deque, Counter


# === Configura tus rutas ===
DATA_ROOT = Path(r"../PASTIS/PASTIS")  # <- cámbiala
IMG_DIR = DATA_ROOT / "DATA_S2"
MSK_DIR = DATA_ROOT / "ANNOTATIONS"
OUT_STATS = Path("pastis_qc/norm_stats.json")
OUT_SPLITS = Path("pastis_qc/splits.csv")
SEED = 13
FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}

# === Config Loader Minimo ====
SPLITS_CSV  = Path("pastis_qc/splits.csv")
STATS_JSON  = Path("pastis_qc/norm_stats.json")
SPLIT_TO_SHOW = "val"     # "train" | "val" | "test"
ID_CHOICE     = None       # p.ej., "10000" para forzar ese ID; None -> toma el primero del split
MODEL_INPUT_SIZE = 256     # entrada sugerida para ViT/DINO (≥256 para patch=16)
OUT_PNG = Path("pastis_qc/paso05_preview.png")
OUT_NPY = Path("pastis_qc/example_input_dino.npy")

# === Config Pytorch Loader ===
SPLIT       = "train"     # "train" | "val" | "test"
INPUT_SIZE  = 256          # ≥256 para ViT patch=16 (anti-"ajedrez")
BATCH_SIZE  = 4
NUM_WORKERS = 0            # súbelo si tu entorno lo permite
BINARIZE_MASK = False      # True -> y = (y>0).astype(int)  (para tarea binaria por-parcela)

# === Config Dino Base ===
EPOCHS      = 2           # smoke run; luego puedes subir
LR_HEAD     = 1e-3
WEIGHT_DECAY= 1e-4
MODEL_CANDS = [
    "facebook/dinov3-vitl16-pretrain-sat493m",  # sat pretraining (pesado)
    "facebook/dinov3-vitb16"                    # fallback más ligero
]

# --- Config visualizar---
OUTPUT_DIR = Path("pastis_qc/vis_val")
N_SAMPLES  = 8          # cuántas muestras de validación visualizar
SPLIT      = "val"

# === Canales Sentinel-2 típicos en PASTIS ===
S2_CHANNELS = [
    "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B11", "B12"
]
# Baseline recomendado: RGB = (B04,B03,B02) → indices (2,1,0)
# Opción 4 bandas: NIR+RGB = (B08,B04,B03,B02) → indices (6,2,1,0)
BASELINE_CHANNELS: Tuple[int, ...] = (2, 1, 0)  # cámbialo a (6,2,1,0) si quieres NIR+RGB

# === Muestreo para estadísticas ===
MAX_FILES = 200          # máximo de parches a usar (sube/baja según tu PC)
PIXELS_PER_FILE = 4096   # píxeles aleatorios por parche (para no cargar todo)
RNG = np.random.default_rng(42)

In [ ]:
# === Utilidades ===
def id_from_name(p: Path, expected_prefix: str) -> str:
    name = p.name
    assert name.startswith(expected_prefix + "_"), f"Nombre inesperado: {name}"
    assert name.endswith(".npy"), f"Extensión inesperada: {name}"
    return name.split("_", 1)[1].rsplit(".", 1)[0]

# Descubrir y emparejar IDs
img_files = sorted((IMG_DIR).glob("S2_*.npy"))
msk_files = sorted((MSK_DIR).glob("ParcelIDs_*.npy"))
img_idx = {id_from_name(p, "S2"): p for p in img_files}
msk_idx = {id_from_name(p, "ParcelIDs"): p for p in msk_files}
common_ids = sorted(set(img_idx) & set(msk_idx))
assert common_ids, "No hay pares imagen+máscara"
ids_used = common_ids[:MAX_FILES]
print(f"[Paso03] Usaré {len(ids_used)} parches para calcular medias/STD de {len(S2_CHANNELS)} bandas.")

# Acumuladores (Welford)
count = 0
mean = np.zeros(len(BASELINE_CHANNELS), dtype=np.float64)
M2   = np.zeros(len(BASELINE_CHANNELS), dtype=np.float64)

def update_welford(x: np.ndarray):
    global count, mean, M2
    # x: (N, Csel) float32
    for row in x:
        count += 1
        delta = row - mean
        mean += delta / count
        M2   += delta * (row - mean)

# Bucle de archivos
for sid in ids_used:
    X = np.load(img_idx[sid])  # (T,C,H,W) típico en tu dataset
    assert X.ndim == 4, f"Espero (T,C,H,W); llegó {X.shape}"
    # Reducimos tiempo (mediana) y pasamos a (H,W,C)
    X_c_hw = np.median(X, axis=0)                 # (C,H,W)
    X_hw_c = np.transpose(X_c_hw, (1, 2, 0)).astype(np.float32)  # (H,W,C)

    H, W, C = X_hw_c.shape
    assert max(BASELINE_CHANNELS) < C, f"Canal fuera de rango, C={C}"

    # Seleccionamos canales y muestreamos píxeles
    X_sel = X_hw_c[:, :, list(BASELINE_CHANNELS)]           # (H,W,Csel)
    N = H * W
    take = min(PIXELS_PER_FILE, N)
    idx = RNG.choice(N, size=take, replace=False)
    X_take = X_sel.reshape(-1, X_sel.shape[-1])[idx]        # (take, Csel)

    update_welford(X_take.astype(np.float64))

# Resultados
if count > 1:
    var = M2 / (count - 1)
    std = np.sqrt(np.maximum(var, 1e-12))
else:
    std = np.ones_like(mean)

stats = {
    "channels_total": S2_CHANNELS,
    "channels_used_indices": list(BASELINE_CHANNELS),
    "channels_used_names": [S2_CHANNELS[i] for i in BASELINE_CHANNELS],
    "mean": mean.tolist(),
    "std": std.tolist(),
    "count_pixels": int(count),
}
OUT_STATS.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)
print(f"[Paso03] Guardé estadísticas en {OUT_STATS}")

# Función para aplicar normalización (z-score) a una imagen (H,W,C)
def apply_norm(x_hw_c: np.ndarray, mean: List[float], std: List[float], channels: Tuple[int, ...]) -> np.ndarray:
    x = x_hw_c[:, :, list(channels)].astype(np.float32)
    m = np.asarray(mean, dtype=np.float32)
    s = np.asarray(std, dtype=np.float32)
    return (x - m) / (s + 1e-6)

# Prueba rápida con un ID
sid0 = ids_used[0]
X0 = np.load(img_idx[sid0])
X0 = np.transpose(np.median(X0, axis=0), (1, 2, 0)).astype(np.float32)
X0n = apply_norm(X0, stats["mean"], stats["std"], tuple(stats["channels_used_indices"]))
print("[Paso03] Ejemplo normalizado:", X0n.shape, X0n.dtype, 
      "→ rango ", float(np.nanmin(X0n)), float(np.nanmax(X0n)))

In [ ]:
# Splits reproducibles (train/val/test)
# Objetivo:
#  - Tomar los IDs comunes (S2_* y ParcelIDs_*)
#  - Partirlos en splits reproducibles con una semilla fija
#  - Guardar un CSV portable con rutas relativa

assert abs(sum(FRACTIONS.values()) - 1.0) < 1e-6, "Las fracciones deben sumar 1.0"
assert IMG_DIR.is_dir(), f"No existe {IMG_DIR}"
assert MSK_DIR.is_dir(), f"No existe {MSK_DIR}"

# Shuffle reproducible
data_ids = np.array(common_ids)
rng = np.random.default_rng(SEED)
rng.shuffle(data_ids)

# Cálculo de cortes
N = len(data_ids)
N_train = int(round(FRACTIONS["train"] * N))
N_val   = int(round(FRACTIONS["val"]   * N))
# Ajuste para que sumen exacto
N_test  = N - N_train - N_val

splits = (
    [("train", i) for i in data_ids[:N_train]] +
    [("val",   i) for i in data_ids[N_train:N_train+N_val]] +
    [("test",  i) for i in data_ids[N_train+N_val:]]
)

# Escribir CSV con rutas relativas a DATA_ROOT
OUT_SPLITS.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_SPLITS, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "split", "img_relpath", "msk_relpath"])  # cabecera
    for split, sid in splits:
        rel_img = img_idx[sid].relative_to(DATA_ROOT).as_posix()
        rel_msk = msk_idx[sid].relative_to(DATA_ROOT).as_posix()
        w.writerow([sid, split, rel_img, rel_msk])

print(f"[Paso04] Escribí {OUT_SPLITS} con {N} filas (train={N_train}, val={N_val}, test={N_test}).")
print("[Paso04] Ejemplo de fila:")
with open(OUT_SPLITS, "r", encoding="utf-8") as f:
    for _ in range(3):
        print("   ", f.readline().strip())

In [ ]:
# Loader mínimo + visual del input para DINOv3
# Objetivo:
#  - Cargar 1 muestra desde splits.csv (según split o ID fijo)
#  - Aplicar reducción temporal (mediana), seleccionar canales, normalizar (stats del Paso 03)
#  - Redimensionar al tamaño de entrada (p.ej., 256×256) para DINOv3 (visual y guardado)
#  - Generar un plot sencillo con: visual RGB y el tensor de entrada que usaremos

try:
    from PIL import Image
    try:
        RESAMPLE_BILINEAR = Image.Resampling.BILINEAR  # Pillow >=9.1
        RESAMPLE_NEAREST  = Image.Resampling.NEAREST
    except Exception:
        RESAMPLE_BILINEAR = Image.BILINEAR
        RESAMPLE_NEAREST  = Image.NEAREST
except Exception as _e:
    Image = None
    print("[WARN] PIL no disponible; se omitirá el resize (usaremos 128×128)")


# Leer splits
rows = []
with open(SPLITS_CSV, "r", encoding="utf-8") as f:
    rdr = csv.DictReader(f)
    for r in rdr:
        rows.append(r)

candidates = [r for r in rows if r["split"] == SPLIT_TO_SHOW]
assert candidates, f"No hay filas con split='{SPLIT_TO_SHOW}' en {SPLITS_CSV}"

row = None
if ID_CHOICE is None:
    row = candidates[0]
else:
    for r in candidates:
        if r["id"] == str(ID_CHOICE):
            row = r
            break
    assert row is not None, f"ID {ID_CHOICE} no encontrado en split {SPLIT_TO_SHOW}"

sid     = row["id"]
img_rel = row["img_relpath"]
msk_rel = row["msk_relpath"]
img_path= DATA_ROOT / img_rel
msk_path= DATA_ROOT / msk_rel

print(f"[Paso05] ID: {sid}")
print("  ", img_path)
print("  ", msk_path)

# Leer stats
with open(STATS_JSON, "r", encoding="utf-8") as f:
    stats = json.load(f)
CH_IDX = tuple(stats["channels_used_indices"])  # p.ej., (2,1,0) o (6,2,1,0)
CH_NAM = stats.get("channels_used_names", [])
MEAN   = stats["mean"]
STD    = stats["std"]

# Cargar datos (T,C,H,W) y máscara (H,W) o (T,H,W)
X = np.load(img_path)   # (T,C,H,W)
y = np.load(msk_path)   # (H,W) o (T,H,W)
assert X.ndim == 4, f"Espero (T,C,H,W); llegó {X.shape}"
C = X.shape[1]

# Reducir tiempo (mediana) y reordenar a (H,W,C)
X_c_hw = np.median(X, axis=0)               # (C,H,W)
X_hw_c = np.transpose(X_c_hw, (1,2,0))      # (H,W,C)

# Si la máscara tuviera tiempo, toma última
if y.ndim == 3:
    y = y[-1]
assert y.ndim == 2, f"Máscara debe ser (H,W); llegó {y.shape}"
assert max(CH_IDX) < X_hw_c.shape[-1], f"Canal fuera de rango; C={X_hw_c.shape[-1]}, CH_IDX={CH_IDX}"

# --- Visual 1: RGB estirado para inspección (si tenemos al menos 3 canales seleccionados) ---
vis_rgb = None
if len(CH_IDX) >= 3:
    vis = X_hw_c[:, :, list(CH_IDX[:3])].astype(np.float32)
    # estirado por percentiles para visual (no para el tensor)
    lo = np.nanpercentile(vis, 2, axis=(0,1)); hi = np.nanpercentile(vis, 98, axis=(0,1))
    vis_rgb = np.clip((vis - lo) / (hi - lo + 1e-6), 0, 1)

# --- Tensor de entrada (normalizado z-score y redimensionado a 256) ---
X_norm = apply_norm(X_hw_c, MEAN, STD, CH_IDX)   # (H,W,Csel)

if Image is not None and MODEL_INPUT_SIZE is not None:
    H, W, Csel = X_norm.shape
    # redimensiona cada canal por separado con bilinear
    resized = []
    for c in range(Csel):
        im = Image.fromarray(X_norm[..., c])
        im = im.resize((MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), resample=RESAMPLE_BILINEAR)
        resized.append(np.array(im, dtype=np.float32))
    X_input = np.stack(resized, axis=-1)  # (S,S,Csel)
    # máscara para overlay (nearest)
    m_im = Image.fromarray(y.astype(np.int32))
    m_im = m_im.resize((MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), resample=RESAMPLE_NEAREST)
    y_res = np.array(m_im, dtype=np.int32)
else:
    X_input = X_norm
    y_res   = y

print("[Paso05] Tensor de entrada para DINOv3 →", X_input.shape, X_input.dtype)
print("Canales usados:", CH_IDX, CH_NAM)

# === Plot sencillo ===
fig, axs = plt.subplots(1, 2 if vis_rgb is not None else 1, figsize=(8, 4))
if not isinstance(axs, np.ndarray):
    axs = np.array([axs])

# Pane 1: Visual RGB (si hay)
if vis_rgb is not None:
    axs[0].imshow(vis_rgb)
    axs[0].set_title("Visual (RGB elegido)")
    axs[0].axis('off')
    ax2 = axs[1]
else:
    ax2 = axs[0]

# Pane 2: Input normalizado (re-escalado 0..1 SOLO para visualizar)
vis_in = X_input.copy()
# escalar por percentiles para que se vea (no afecta el tensor guardado)
lo = np.nanpercentile(vis_in, 2, axis=(0,1)); hi = np.nanpercentile(vis_in, 98, axis=(0,1))
vis_in = np.clip((vis_in - lo) / (hi - lo + 1e-6), 0, 1)
if vis_in.shape[-1] >= 3:
    ax2.imshow(vis_in[..., :3])
else:
    ax2.imshow(vis_in[..., 0], cmap='gray')
ax2.set_title(f"Input DINOv3 ({X_input.shape[0]}×{X_input.shape[1]}×{X_input.shape[2]})")
ax2.axis('off')

fig.tight_layout()
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=150)
plt.close(fig)
print(f"[Paso05] Guardé previsualización en {OUT_PNG}")

# Guardar el tensor de entrada como .npy (para reutilizar después)
np.save(OUT_NPY, X_input.astype(np.float32))
print(f"[Paso05] Guardé tensor en {OUT_NPY}")


In [ ]:
# Dataset/Loader (PyTorch) + Smoke test
# Objetivo:
#  - Dataset que lea splits.csv y norm_stats.json
#  - Para cada muestra: (T,C,H,W) -> reducir tiempo (mediana) -> (H,W,C)
#    -> seleccionar canales -> normalizar -> redimensionar a SxS
#  - Devolver tensores listos (X: float32 [C,S,S], y: long [S,S])
#  - Smoke test: 1 item y 1 batch

class PastisPatchDataset(Dataset):
    def __init__(self,
                 data_root: Path,
                 splits_csv: Path,
                 stats_json: Path,
                 split: str = "train",
                 input_size: int = 256,
                 binarize_mask: bool = False,
                 temporal_reduce: str = "median"):
        super().__init__()
        self.data_root = Path(data_root)
        self.input_size = int(input_size)
        self.binarize_mask = bool(binarize_mask)
        self.temporal_reduce = temporal_reduce

        # Lee splits
        rows = []
        with open(splits_csv, "r", encoding="utf-8") as f:
            rdr = csv.DictReader(f)
            for r in rdr:
                if r["split"] == split:
                    rows.append(r)
        assert rows, f"No hay filas para split='{split}' en {splits_csv}"
        self.rows = rows

        # Lee stats
        with open(stats_json, "r", encoding="utf-8") as f:
            st = json.load(f)
        self.ch_idx: Tuple[int, ...] = tuple(st["channels_used_indices"])  # p.ej., (2,1,0) o (6,2,1,0)
        self.mean = st["mean"]
        self.std  = st["std"]

    def __len__(self):
        return len(self.rows)

    def _reduce_time(self, X: np.ndarray) -> np.ndarray:
        # X: (T,C,H,W)
        if self.temporal_reduce == "last":
            X_c_hw = X[-1]
        elif self.temporal_reduce == "mean":
            X_c_hw = X.mean(axis=0)
        else:  # median (robusto)
            X_c_hw = np.median(X, axis=0)
        return X_c_hw  # (C,H,W)

    def _resize_tensor(self, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # X: (H,W,Csel), y: (H,W)
        if Image is None or self.input_size is None:
            return X, y
        H, W, Csel = X.shape
        # redimensionamos canal por canal (bilinear) y máscara (nearest)
        xs = []
        for c in range(Csel):
            im = Image.fromarray(X[..., c])
            im = im.resize((self.input_size, self.input_size), resample=RESAMPLE_BILINEAR)
            xs.append(np.array(im, dtype=np.float32))
        Xr = np.stack(xs, axis=-1)
        ym = Image.fromarray(y.astype(np.int32))
        ym = ym.resize((self.input_size, self.input_size), resample=RESAMPLE_NEAREST)
        yr = np.array(ym, dtype=np.int32)
        return Xr, yr

    def __getitem__(self, i: int) -> Dict[str, torch.Tensor]:
        r = self.rows[i]
        img_path = self.data_root / r["img_relpath"]
        msk_path = self.data_root / r["msk_relpath"]

        X = np.load(img_path)   # (T,C,H,W)
        y = np.load(msk_path)   # (H,W) o (T,H,W)
        if y.ndim == 3:
            y = y[-1]
        # reduce tiempo y reordena a (H,W,C)
        X_c_hw = self._reduce_time(X)          # (C,H,W)
        X_hw_c = np.transpose(X_c_hw, (1,2,0)) # (H,W,C)

        # binariza si aplica
        if self.binarize_mask:
            y = (y > 0).astype(np.int64)
        else:
            y = y.astype(np.int64)

        # normaliza y selecciona canales
        X_norm = apply_norm(X_hw_c, self.mean, self.std, self.ch_idx)  # (H,W,Csel)
        # redimensiona a SxS
        X_in, y_in = self._resize_tensor(X_norm, y)

        # a tensores PyTorch
        X_t = torch.from_numpy(X_in).permute(2,0,1).contiguous().float()  # (C,S,S)
        y_t = torch.from_numpy(y_in).long()                                # (S,S)
        return {"image": X_t, "mask": y_t, "id": r["id"]}

# --- Smoke test ---
if __name__ == "__main__":
    ds = PastisPatchDataset(DATA_ROOT, SPLITS_CSV, STATS_JSON, split=SPLIT,
                            input_size=INPUT_SIZE, binarize_mask=BINARIZE_MASK)
    print(f"[Paso06] Dataset {SPLIT}: {len(ds)} muestras; canales usados: {len(ds.ch_idx)} → {ds.ch_idx}")

    ex = ds[0]
    print("[Paso06] Item 0 -> X:", tuple(ex["image"].shape), ex["image"].dtype,
          "| y:", tuple(ex["mask"].shape), ex["mask"].dtype, "| id:", ex["id"]) 

    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    batch = next(iter(dl))
    Xb, yb = batch["image"], batch["mask"]
    print("[Paso06] Batch -> X:", tuple(Xb.shape), Xb.dtype, "| y:", tuple(yb.shape), yb.dtype)

In [ ]:
# ==============================================
# PASO 07 – Baseline con DINOv3 congelado + head ligero (binario)
# ==============================================
# Objetivo:
#  - Entrenar un baseline rápido anti-"ajedrez": backbone DINOv3 congelado
#    + decoder ligero con upsampling aprendido. Tarea binaria: "parcela vs fondo"
#  - Métricas: pérdida y IoU de la clase 1 (parcela)
#  - Nota: requiere haber ejecutado PASO 06 (Dataset) o que la clase PastisPatchDataset esté definida


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Dataset (usa la clase del PASO 06) ===
try:
    _ = PastisPatchDataset
except NameError:
    raise RuntimeError("No se encontró PastisPatchDataset. Ejecuta/define el PASO 06 en este mismo notebook.")

train_ds = PastisPatchDataset(DATA_ROOT, SPLITS_CSV, STATS_JSON, split="train",
                              input_size=INPUT_SIZE, binarize_mask=True)
val_ds   = PastisPatchDataset(DATA_ROOT, SPLITS_CSV, STATS_JSON, split="val",
                              input_size=INPUT_SIZE, binarize_mask=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# === Modelo ===
class DinoV3Seg(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 2, input_size: int = 256):
        super().__init__()
        self.backbone = backbone.eval()  # congelado al inicio
        for p in self.backbone.parameters():
            p.requires_grad = False
        cfg = getattr(self.backbone, "config", None)
        hdim = getattr(cfg, "hidden_size", None)
        if hdim is None:
            raise RuntimeError("No pude leer hidden_size de la config del backbone")
        self.nreg = int(getattr(cfg, "num_register_tokens", 0))
        self.patch = int(getattr(cfg, "patch_size", 16))
        # head ligero: 2 etapas de upsampling aprendido
        self.head = nn.Sequential(
            nn.Conv2d(hdim, hdim // 2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hdim // 2, num_classes, kernel_size=1),
        )
        self.input_size = input_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,S,S) ya normalizado y redimensionado a S=INPUT_SIZE
        with torch.no_grad():
            out = self.backbone(pixel_values=x)  # last_hidden_state: (B, 1+nreg+Npatch, hdim)
            tokens = out.last_hidden_state[:, 1 + self.nreg :, :]
        B, N, C = tokens.shape
        Ht = Wt = int(math.sqrt(N))  # válido al usar S y patch fijos
        feat = tokens.transpose(1, 2).reshape(B, C, Ht, Wt)  # (B,C,Ht,Wt)
        logits = self.head(feat)  # (B,K,Ht,Wt)
        logits_up = F.interpolate(logits, size=(x.shape[-2], x.shape[-1]), mode="bilinear", align_corners=False)
        return logits_up  # (B,K,S,S)

# Cargar backbone (probando candidatos)
loaded = None
last_err = None
for mid in MODEL_CANDS:
    try:
        print(f"[Paso07] Cargando backbone: {mid}")
        bb = AutoModel.from_pretrained(mid)
        loaded = (mid, bb)
        break
    except Exception as e:
        print(f"[Paso07] Falló {mid}: {e}")
        last_err = e

if loaded is None:
    raise RuntimeError(f"No pude cargar ningún backbone. Último error: {last_err}")

MODEL_ID, BACKBONE = loaded
model = DinoV3Seg(BACKBONE, num_classes=2, input_size=INPUT_SIZE).to(DEVICE)
print(f"[Paso07] Usando {MODEL_ID} | patch={model.patch} | nreg={model.nreg}")

# === Entrenamiento ===
optim = torch.optim.AdamW(model.head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()  # y: 0=fondo, 1=parcela

@torch.no_grad()
def eval_loop(dl) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    inter = 0
    union = 0
    n = 0
    for batch in dl:
        X = batch["image"].to(DEVICE)  # (B,C,S,S)
        y = batch["mask"].to(DEVICE)   # (B,S,S)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += float(loss.item())
        # IoU clase 1
        preds = torch.argmax(logits, dim=1)  # (B,S,S)
        pred1 = (preds == 1)
        targ1 = (y == 1)
        inter += torch.logical_and(pred1, targ1).sum().item()
        union += torch.logical_or(pred1, targ1).sum().item()
        n += 1
    miou = (inter / union) if union > 0 else 0.0
    return {"loss": total_loss / max(n,1), "iou1": miou}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_dl:
        X = batch["image"].to(DEVICE)
        y = batch["mask"].to(DEVICE)
        logits = model(X)
        loss = criterion(logits, y)
        optim.zero_grad(set_to_none=True)
        loss.backward()
        optim.step()
        epoch_loss += float(loss.item())
    tr_loss = epoch_loss / max(1, len(train_dl))
    val_metrics = eval_loop(val_dl)
    print(f"[Paso07][Ep {epoch}/{EPOCHS}] train_loss={tr_loss:.4f} | val_loss={val_metrics['loss']:.4f} | val_IoU(class=1)={val_metrics['iou1']:.3f}")

print("[Paso07] Listo. Tienes un baseline binario con DINOv3 congelado.")
print("Siguientes pasos: descongelar últimos bloques, pasar a multiclase (si tienes labels), y sliding-window en inferencia.")

In [ ]:
# -----------------
# Modelo (igual esquema del Paso 07)
# -----------------
print(f"[Paso08] Cargando backbone: {MODEL_ID}")
backbone = AutoModel.from_pretrained(MODEL_ID)
model = DinoV3Seg(backbone, num_classes=2, input_size=INPUT_SIZE).to(DEVICE)

# -----------------
# Detección robusta de bloques del ViT y descongelado
# -----------------

def find_blocklists(module: nn.Module, min_len: int = 6):
    """Devuelve [(path, modulelist, score)] candidatas a ser la lista de bloques del ViT.
    score alto si el primer bloque contiene submódulos de attn/mlp/norm.
    """
    out = []
    q = deque([("", module)])
    while q:
        base, m = q.popleft()
        for name, child in m.named_children():
            path = f"{base}.{name}" if base else name
            if isinstance(child, nn.ModuleList) and len(child) >= min_len:
                score = 0
                first = child[0]
                for subn, _ in first.named_modules():
                    s = subn.lower()
                    if any(k in s for k in ("attn", "attention")): score += 1
                    if any(k in s for k in ("mlp", "ffn", "feedforward")): score += 1
                    if any(k in s for k in ("norm", "layernorm", "ln")): score += 1
                out.append((path, child, score))
            q.append((path, child))
    out.sort(key=lambda t: (t[2], len(t[1])), reverse=True)
    return out

# 1) congela todo
for p in model.backbone.parameters():
    p.requires_grad = False

# 2) busca la lista de bloques y toma los últimos K
candidates = find_blocklists(model.backbone, min_len=6)
if not candidates:
    raise RuntimeError("No pude detectar la lista de bloques del ViT en el backbone.")
blocks_path, blocks_list, score = candidates[0]
num_blocks = len(blocks_list)
k = max(1, min(UNFREEZE_LAST_K, num_blocks))
last_idxs = list(range(num_blocks - k, num_blocks))

for idx in last_idxs:
    for n, p in blocks_list[idx].named_parameters():
        p.requires_grad = True

# 3) permite grad en normalizaciones globales si existen
for nm, p in model.backbone.named_parameters():
    if any(tag in nm.lower() for tag in ["ln_f", "layernorm", "post_layernorm", "norm.weight", "norm.bias"]):
        p.requires_grad = True

unfrozen = [n for n,p in model.backbone.named_parameters() if p.requires_grad]
print(f"[Paso08] Bloques detectados en: {blocks_path} (len={num_blocks}, score={score})")
print(f"[Paso08] Descongelados últimos {k} bloques: idx={last_idxs}")
print(f"[Paso08] Parámetros con grad: {len(unfrozen)}")

# -----------------
# Optimizador con dos grupos de LR
# -----------------
head_params = list(model.head.parameters())
backbone_ft_params = [p for p in model.backbone.parameters() if p.requires_grad]
optim = torch.optim.AdamW(
    [
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
        {"params": backbone_ft_params, "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
    ]
)
criterion = nn.CrossEntropyLoss()

# AMP opcional
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def eval_loop(dl) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    inter = union = n = 0
    for batch in dl:
        X = batch["image"].to(DEVICE)
        y = batch["mask"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(X)
            loss = criterion(logits, y)
        total_loss += float(loss.item())
        preds = torch.argmax(logits, dim=1)
        pred1 = (preds == 1)
        targ1 = (y == 1)
        inter += torch.logical_and(pred1, targ1).sum().item()
        union += torch.logical_or(pred1, targ1).sum().item()
        n += 1
    miou = (inter / union) if union > 0 else 0.0
    return {"loss": total_loss / max(n,1), "iou1": miou}

# -----------------
# Entrenamiento corto
# -----------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for batch in train_dl:
        X = batch["image"].to(DEVICE)
        y = batch["mask"].to(DEVICE)
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(X)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
        running += float(loss.item())
    tr_loss = running / max(1, len(train_dl))
    val_metrics = eval_loop(val_dl)
    print(f"[Paso08][Ep {epoch}/{EPOCHS}] train_loss={tr_loss:.4f} | val_loss={val_metrics['loss']:.4f} | val_IoU(class=1)={val_metrics['iou1']:.3f}")

print("[Paso08] Fine-tuning parcial completo. Compara IoU con Paso 07.")

In [ ]:
# Visualización de resultados (val) + métricas por muestra
# Objetivo:
#  - Generar overlays para varias muestras de validación
#  - Guardar PNGs (input RGB aprox + GT + prob mapa + pred overlay)
#  - Guardar CSV con IoU por muestra (clase=1)
#  - Requiere: modelo `model` ya entrenado (Paso 07/08) y el DataLoader de val


# --- Validaciones mínimas ---
assert 'model' in globals(), "No hay un modelo `model` en memoria. Ejecuta Paso 07/08 antes."
model.eval()

# Reintenta obtener el val_dl si no existe
if 'val_dl' not in globals():
    assert 'PastisPatchDataset' in globals(), "Falta el Dataset (Paso 06)."
    from torch.utils.data import DataLoader
    DATA_ROOT   = Path(r"/ruta/a/PASTIS/PASTIS")  # ajusta si es necesario
    SPLITS_CSV  = Path("pastis_qc/splits.csv")
    ds_val = PastisPatchDataset(DATA_ROOT, SPLITS_CSV, STATS_JSON, split=SPLIT,
                                input_size=256, binarize_mask=True)
    val_dl = DataLoader(ds_val, batch_size=4, shuffle=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cargar stats para des-normalizar a modo visual
with open(STATS_JSON, "r", encoding="utf-8") as f:
    stats = json.load(f)
MEAN = np.asarray(stats["mean"], dtype=np.float32)
STD  = np.asarray(stats["std"], dtype=np.float32)
CH_IDX = tuple(stats["channels_used_indices"])  # p.ej., (2,1,0)

# --- Utilidades de visual ---

def denorm_visual(x_t: torch.Tensor) -> np.ndarray:
    """x_t: (C,S,S) normalizado; devuelve (S,S,3) para visual (usa los 3 primeros canales si hay >3)."""
    x = x_t.detach().cpu().numpy()  # (C,S,S)
    Csel, S, _ = x.shape
    # des-normalizar cada canal según stats
    x_den = np.empty_like(x)
    for i in range(Csel):
        x_den[i] = x[i] * STD[i] + MEAN[i]
    x_den = np.transpose(x_den, (1,2,0))  # (S,S,Csel)
    # si hay más de 3, toma los 3 primeros para RGB aproximado
    if x_den.shape[-1] >= 3:
        img = x_den[..., :3]
    else:
        img = np.repeat(x_den[..., :1], 3, axis=-1)
    # estirado por percentiles para visualización
    lo = np.nanpercentile(img, 2, axis=(0,1)); hi = np.nanpercentile(img, 98, axis=(0,1))
    img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return img

@torch.no_grad()
def collect_samples(dl, n: int):
    """Devuelve listas (imgs, gts, probs, preds, ids) con hasta n elementos."""
    imgs = []; gts = []; probs = []; preds = []; ids = []
    for batch in dl:
        X = batch["image"].to(next(model.parameters()).device)
        y = batch["mask"].cpu().numpy()
        logits = model(X)                          # (B,2,S,S)
        p = F.softmax(logits, dim=1)[:,1]          # prob clase=1 (B,S,S)
        pr = torch.argmax(logits, dim=1)           # (B,S,S)
        for i in range(X.shape[0]):
            imgs.append(denorm_visual(X[i]))       # (S,S,3)
            gts.append(y[i])                       # (S,S)
            probs.append(p[i].detach().cpu().numpy())
            preds.append(pr[i].detach().cpu().numpy())
            ids.append(batch["id"][i])
            if len(imgs) >= n:
                return imgs, gts, probs, preds, ids
    return imgs, gts, probs, preds, ids

# --- Colectar muestras ---
imgs, gts, probs, preds, ids = collect_samples(val_dl, N_SAMPLES)
print(f"[Paso09] Visualizando {len(imgs)} muestras del split '{SPLIT}'")

# --- Métricas por muestra y guardado de PNGs ---
rows = [("id", "iou1", "tp", "fp", "fn", "pixels")]

def iou_binary(pred: np.ndarray, gt: np.ndarray) -> tuple:
    pred1 = (pred == 1)
    gt1   = (gt == 1)
    inter = np.logical_and(pred1, gt1).sum()
    union = np.logical_or(pred1, gt1).sum()
    iou = float(inter) / float(union) if union > 0 else 0.0
    tp = np.logical_and(pred1, gt1).sum()
    fp = np.logical_and(pred1, ~gt1).sum()
    fn = np.logical_and(~pred1, gt1).sum()
    return iou, int(tp), int(fp), int(fn), int(gt.size)

for i in range(len(imgs)):
    img = imgs[i]
    gt  = gts[i]
    pr  = preds[i]
    pb  = probs[i]
    sid = ids[i]

    iou, tp, fp, fn, npx = iou_binary(pr, gt)
    rows.append((sid, f"{iou:.4f}", tp, fp, fn, npx))

    # Figura con 4 paneles: input, GT, prob, pred overlay
    fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
    ax[0].imshow(img); ax[0].set_title(f"ID {sid}Input (RGB aprox)") ; ax[0].axis('off')
    ax[1].imshow(gt, cmap='gray'); ax[1].set_title("GT (binaria)") ; ax[1].axis('off')
    ax[2].imshow(pb, vmin=0, vmax=1, cmap='turbo'); ax[2].set_title("Prob clase=1") ; ax[2].axis('off')
    # overlay pred
    ax[3].imshow(img)
    ax[3].imshow(pr, cmap='Reds', alpha=0.35)
    ax[3].set_title(f"Pred (IoU={iou:.2f})") ; ax[3].axis('off')
    fig.tight_layout()
    out_png = OUTPUT_DIR / f"val_{sid}.png"
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

# Guardar CSV de métricas por muestra
out_csv = OUTPUT_DIR / "val_metrics.csv"
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerows(rows)

print(f"[Paso09] PNGs guardados en {OUTPUT_DIR}")
print(f"[Paso09] Métricas por muestra → {out_csv}")